# Neural Network Classroom Game — *Plant Doctor Challenge*

In this activity, the class plays the role of a small neural network trained to do something a plant pathologist does every day: combine several imperfect diagnostic signals into a single call — **Healthy** or **Sick**.

The model receives four measurements from a field or greenhouse sample:

- **SNP Diversity Index** — a normalized (0–1) genotyping-based proxy for genetic diversity at known resistance loci. Populations with more diversity at these loci tend to carry more durable resistance.
- **Pathogen Load** — a normalized (0–1) inoculum titer (e.g., relative qPCR or ELISA reading). Higher values indicate greater pathogen pressure in planta.
- **Leaf Color Index** — a normalized (0–1) chlorophyll/canopy-greenness proxy (think: a scaled SPAD reading). Lower values indicate chlorosis consistent with stress or infection.
- **Root Biomass** — normalized (0–1) root dry weight relative to a healthy control. Many pathogens systemically reduce root vigor even when foliar symptoms are subtle.

Just like a diagnostic model doesn't get to see the ground truth in advance, your network doesn't either — it only sees these four numbers and has to produce a probability of disease.

**Roles (for in-class play):**
- **Coach**: knows the true, lab-confirmed diagnosis and revises the weights & bias after each attempt — this is the classroom stand-in for *backpropagation*.
- **Input Neuron**: reads off the four measurements for the chosen sample.
- **Hidden Neuron**: computes the weighted sum `z` and applies the sigmoid to turn it into a probability.
- **Output Neuron**: thresholds the probability into a final call (Healthy/Sick).

**A few translations, since you already think like plant pathologists:**

| ML term | Plant-pathology equivalent |
|---|---|
| Weight (`w`) | How heavily you personally weigh a given symptom/measurement when forming a diagnosis |
| Bias (`b`) | Your baseline prior — e.g., background disease prevalence in the field, before looking at this sample's numbers at all |
| Sigmoid `σ(z)` | Converting several combined risk factors into one bounded (0–1) disease-probability score, much like a composite disease severity index |
| Coach correction | Recalibrating your diagnostic weighting after checking a call against a lab-confirmed (ground-truth) diagnosis |

Run the cells in order. You'll pick a sample, act as the neurons to get a probability, see whether the call was right, and then play Coach to correct the weights — either by typing your own numbers, or, in this version, by asking the Coach's Calculator to compute the correction via one step of gradient descent.

## Step 0: Choose a Sample

Four accessions/samples are pre-loaded below, each with real lab-confirmed diagnoses. They range from an ambiguous field sample to a clear healthy control, a clearly diseased inoculated line, and a *tolerant* line — included on purpose, since tolerance (staying relatively healthy despite measurable pathogen load) is a distinct concept from resistance and makes for a good model-behavior discussion.

| Sample | SNP Diversity | Pathogen Load | Leaf Color | Root Biomass | Confirmed Diagnosis |
|---|:---:|:---:|:---:|:---:|:---:|
| Accession 118 — Field Trial Block 3 | 0.18 | 0.20 | 0.40 | 0.60 | Sick (0) |
| Accession 42 — Greenhouse Control | 0.75 | 0.05 | 0.85 | 0.80 | Healthy (1) |
| Accession 7 — Inoculated, 14 DPI | 0.30 | 0.65 | 0.25 | 0.35 | Sick (0) |
| Accession 91 — Putative Tolerant Line | 0.55 | 0.30 | 0.55 | 0.50 | Healthy (1) |

> The **Coach** picks the sample from the dropdown in the next cell. Everything below (features, true label, error, and the History table) automatically follows that selection.

In [1]:
#@title Setup: samples and helpers (run this cell first)
%matplotlib inline

from math import exp
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

FEATURE_NAMES = ["SNP Diversity Index", "Pathogen Load", "Leaf Color Index", "Root Biomass"]
FEATURE_SHORT = ["SNP", "Path", "Leaf", "Root"]

# Pre-loaded samples. Instructors: edit/add rows here to bring your own data.
PLANTS = {
    "Accession 118 — Field Trial Block 3": {
        "features": {"SNP Diversity Index": 0.18, "Pathogen Load": 0.20, "Leaf Color Index": 0.40, "Root Biomass": 0.60},
        "label": 0,
    },
    "Accession 42 — Greenhouse Control": {
        "features": {"SNP Diversity Index": 0.75, "Pathogen Load": 0.05, "Leaf Color Index": 0.85, "Root Biomass": 0.80},
        "label": 1,
    },
    "Accession 7 — Inoculated, 14 DPI": {
        "features": {"SNP Diversity Index": 0.30, "Pathogen Load": 0.65, "Leaf Color Index": 0.25, "Root Biomass": 0.35},
        "label": 0,
    },
    "Accession 91 — Putative Tolerant Line": {
        "features": {"SNP Diversity Index": 0.55, "Pathogen Load": 0.30, "Leaf Color Index": 0.55, "Root Biomass": 0.50},
        "label": 1,
    },
}

def sigmoid(z):
    return 1.0 / (1.0 + exp(-z))

def label_text(v):
    return "Healthy (1)" if v == 1 else "Sick (0)"

# --- Network diagram -------------------------------------------------------
GRAY = "#C3C9D1"      # not reached yet
BLUE = "#2E5B8A"      # already completed
GOLD = "#C9971F"      # current step, active now
GREEN = "#1a7f37"
RED = "#c0392b"

STAGE_TITLES = {
    -1: "Network ready — click Step 1 to begin",
    0: "Step 1: Input Neuron reads the measurements",
    1: "Step 2: Hidden Neuron computes the weighted sum  z = w·x + b",
    2: "Step 3: Hidden Neuron applies the sigmoid  σ(z)",
    3: "Step 4: Output Neuron makes the call",
}

def _stage_color(elem_stage, stage):
    if elem_stage > stage:
        return GRAY
    elif elem_stage == stage:
        return GOLD
    return BLUE

def draw_network(stage, plant_name="", vals=None, ws=None, bias=None,
                  z=None, prob=None, pred=None, true_label=None):
    fig, ax = plt.subplots(figsize=(9.8, 4.8), dpi=110)
    ax.set_xlim(0, 10.6)
    ax.set_ylim(0, 8)
    ax.axis("off")
    ax.set_aspect("equal")

    in_x = 1.1
    in_y = [6.4, 4.8, 3.2, 1.6]
    sum_x, sum_y = 4.7, 3.9
    bias_x, bias_y = 4.7, 0.6
    act_x, act_y = 7.3, 3.9
    out_x, out_y = 9.6, 3.9

    c_input = _stage_color(0, stage)
    c_sum = _stage_color(1, stage)
    c_act = _stage_color(2, stage)
    if stage >= 3 and true_label is not None and pred is not None:
        c_out = GREEN if pred == true_label else RED
    else:
        c_out = _stage_color(3, stage)

    # edges: inputs -> sum
    for i in range(4):
        ax.annotate("", xy=(sum_x - 0.65, sum_y + (in_y[i]-sum_y)*0.12), xycoords='data',
                     xytext=(in_x + 0.55, in_y[i]), textcoords='data',
                     arrowprops=dict(arrowstyle='-|>', color=c_sum, lw=2.2 if stage == 1 else 1.4))
        if ws is not None:
            mx, my = (in_x + sum_x) / 2 - 0.2, (in_y[i] + sum_y) / 2 + 0.15
            ax.text(mx, my, f"w{i+1}={ws[i]:g}", fontsize=8, color=c_sum, ha='center')

    # bias -> sum
    ax.annotate("", xy=(sum_x, sum_y - 0.62), xycoords='data',
                 xytext=(bias_x, bias_y + 0.42), textcoords='data',
                 arrowprops=dict(arrowstyle='-|>', color=c_sum, lw=2.2 if stage == 1 else 1.4))
    ax.add_patch(Circle((bias_x, bias_y), 0.38, facecolor='white', edgecolor=c_sum, lw=2, zorder=3))
    ax.text(bias_x, bias_y, f"bias\n{bias:g}" if bias is not None else "bias", fontsize=7.5, ha='center', va='center', zorder=4)

    # sum -> activation
    ax.annotate("", xy=(act_x - 0.62, act_y), xycoords='data',
                 xytext=(sum_x + 0.62, sum_y), textcoords='data',
                 arrowprops=dict(arrowstyle='-|>', color=c_act, lw=2.2 if stage == 2 else 1.4))

    # activation -> output
    ax.annotate("", xy=(out_x - 0.62, out_y), xycoords='data',
                 xytext=(act_x + 0.62, act_y), textcoords='data',
                 arrowprops=dict(arrowstyle='-|>', color=c_out, lw=2.2 if stage == 3 else 1.4))

    # input nodes
    for i in range(4):
        ax.add_patch(Circle((in_x, in_y[i]), 0.55, facecolor='white', edgecolor=c_input, lw=2.5, zorder=3))
        label = FEATURE_SHORT[i]
        if vals is not None:
            label += f"\n{vals[i]:g}"
        ax.text(in_x, in_y[i], label, fontsize=8.5, ha='center', va='center', zorder=4)

    # sum node
    ax.add_patch(Circle((sum_x, sum_y), 0.62, facecolor='white', edgecolor=c_sum, lw=2.8, zorder=3))
    ax.text(sum_x, sum_y, "Σ", fontsize=20, ha='center', va='center', zorder=4, color=c_sum)
    if stage >= 1 and z is not None:
        ax.text(sum_x, sum_y - 0.95, f"z = {z:.4f}", fontsize=9, ha='center', color=BLUE, fontweight='bold')

    # activation node
    ax.add_patch(Circle((act_x, act_y), 0.62, facecolor='white', edgecolor=c_act, lw=2.8, zorder=3))
    ax.text(act_x, act_y, "σ", fontsize=20, ha='center', va='center', zorder=4, color=c_act)
    if stage >= 2 and prob is not None:
        ax.text(act_x, act_y - 0.95, f"σ(z) = {prob:.4f}", fontsize=9, ha='center', color=BLUE, fontweight='bold')

    # output node
    ax.add_patch(Circle((out_x, out_y), 0.62, facecolor='white', edgecolor=c_out, lw=2.8, zorder=3))
    out_symbol = "?"
    if stage >= 3 and pred is not None:
        out_symbol = "H" if pred == 1 else "S"
    ax.text(out_x, out_y, out_symbol, fontsize=20, ha='center', va='center', zorder=4,
             color=c_out, fontweight='bold')
    if stage >= 3 and pred is not None:
        ax.text(out_x, out_y - 0.95, label_text(pred), fontsize=9, ha='center', color=c_out, fontweight='bold')

    # labels under each column
    ax.text(in_x, 0.15, "Input\nNeuron", fontsize=8, ha='center', color='#555')
    ax.text(sum_x, 7.4, "Hidden Neuron", fontsize=9, ha='center', color='#555')
    ax.text(act_x, 0.15, "Activation\n(sigmoid)", fontsize=8, ha='center', color='#555')
    ax.text(out_x, 0.15, "Output\nNeuron", fontsize=8, ha='center', color='#555')

    title = STAGE_TITLES.get(stage, "")
    subtitle = f"Sample: {plant_name}" if plant_name else ""
    ax.set_title(f"{title}\n{subtitle}", fontsize=10.5, loc='left', color='#1F3B57')

    plt.tight_layout()
    return fig

plant_dropdown = widgets.Dropdown(
    options=list(PLANTS.keys()),
    value=list(PLANTS.keys())[0],
    description="Sample:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)
plant_out = widgets.Output()

def render_plant(_=None):
    plant_out.clear_output()
    p = PLANTS[plant_dropdown.value]
    df = pd.DataFrame({
        "Feature": FEATURE_NAMES,
        "Value": [p["features"][f] for f in FEATURE_NAMES],
    })
    with plant_out:
        display(df.style.hide(axis="index"))
        display(HTML(f"<b>Confirmed diagnosis:</b> {label_text(p['label'])}"))

plant_dropdown.observe(render_plant, names="value")
display(plant_dropdown, plant_out)
render_plant()

HTML("<b>Setup complete.</b> A sample is selected above. Scroll down to act as the neurons.")

Dropdown(description='Sample:', layout=Layout(width='420px'), options=('Accession 118 — Field Trial Block 3', …

Output()

## Step 1–3: Act like neurons

- **Input Neuron**: read the four measurements for the selected sample.
- **Hidden Neuron**: multiply each by its weight and add the bias → `z = w·x + b`.
- Apply the **sigmoid** to turn `z` into a disease probability, `σ(z)`.
- **Output Neuron**: if `σ(z) ≥ 0.5` → call it **Healthy (1)**, otherwise **Sick (0)**.

The diagram below is the same neuron layout from lecture: four input nodes → a weighted-sum (`Σ`) node → an activation (`σ`) node → an output node. Click **Step 1 / 2 / 3 / 4** one at a time — each click lights up (gold) the node and connections being computed *right now*, keeps the already-computed part highlighted in blue, and leaves what's still ahead in gray, so the class can always point to "where we are" in the calculation. Compare the call to the confirmed diagnosis and note the **Error** at Step 4 — that gap is exactly what the Coach's correction (Step 5 below) is trying to close.

In [2]:
#@title Step through the forward pass (play the game!)
# Widgets for weights & bias (the Coach can change these)
w1 = widgets.FloatText(value=0.0, description='w1 (SNP):')
w2 = widgets.FloatText(value=0.0, description='w2 (Path):')
w3 = widgets.FloatText(value=0.0, description='w3 (Leaf):')
w4 = widgets.FloatText(value=0.0, description='w4 (Root):')
b1 = widgets.FloatText(value=0.0, description='Bias (B1):')

STEP_LABELS = [
    '▶ Step 1: Read Inputs',
    '▶ Step 2: Weighted Sum (z)',
    '▶ Step 3: Activation σ(z)',
    '▶ Step 4: Output & Decision',
]

step_button = widgets.Button(description=STEP_LABELS[0], button_style='primary',
                              layout=widgets.Layout(width='260px'))
reset_button = widgets.Button(description='Reset weights & history', button_style='warning')
out = widgets.Output()

history = []          # one dict per completed round (all four steps)
stage = {"value": -1}  # -1 = nothing shown yet, 0..3 = current step

def render_stage(s):
    plant_name = plant_dropdown.value
    p = PLANTS[plant_name]
    vals = [p["features"][f] for f in FEATURE_NAMES]
    ws = [w1.value, w2.value, w3.value, w4.value]
    bias = b1.value
    true_label = p["label"]

    prods = [vals[i] * ws[i] for i in range(4)]
    z = sum(prods) + bias
    prob = sigmoid(z)
    pred = 1 if prob >= 0.5 else 0
    error = true_label - prob
    correct = (pred == true_label)

    fig = draw_network(
        s, plant_name=plant_name, vals=vals, ws=ws, bias=bias,
        z=z if s >= 1 else None,
        prob=prob if s >= 2 else None,
        pred=pred if s >= 3 else None,
        true_label=true_label if s >= 3 else None,
    )

    out.clear_output(wait=True)
    with out:
        display(fig)
        plt.close(fig)
        if s >= 1:
            calc_df = pd.DataFrame({
                "Feature": FEATURE_NAMES + ["Bias"],
                "Value": vals + ["—"],
                "Weight": ws + ["B1"],
                "Value × Weight": [round(x, 6) for x in prods] + [bias],
            })
            display(calc_df.style.hide(axis='index'))
            print(f"z = {z:.6f}")
        if s >= 2:
            print(f"σ(z) = {prob:.6f}")
        if s >= 3:
            print(f"Prediction: {label_text(pred)}   |   Confirmed diagnosis: {label_text(true_label)}")
            print(f"Error (True − σ(z)) = {error:.6f}")
            if correct:
                display(HTML("<h3 style='color:#1a7f37;'>✅ Correct diagnosis!</h3>"))
            else:
                display(HTML("<h3 style='color:#c0392b;'>❌ Misdiagnosis — Coach, please correct the weights (Step 5 below).</h3>"))
            history.append({
                "Round": len(history) + 1, "Sample": plant_name,
                "w1": ws[0], "w2": ws[1], "w3": ws[2], "w4": ws[3], "Bias": bias,
                "z": round(z, 4), "sigma(z)": round(prob, 4),
                "Prediction": label_text(pred), "True Label": label_text(true_label),
                "Error": round(error, 4), "Correct?": "✅" if correct else "❌",
            })
            display(HTML("<b>History (all completed rounds this session):</b>"))
            display(pd.DataFrame(history).style.hide(axis='index'))

def run_step(_):
    s = stage["value"]
    s = 0 if s < 0 or s >= 3 else s + 1
    stage["value"] = s
    render_stage(s)
    step_button.description = ('↻ New Round — ' + STEP_LABELS[0]) if s == 3 else STEP_LABELS[s + 1]

def reset_all(_):
    for w in (w1, w2, w3, w4, b1):
        w.value = 0.0
    history.clear()
    stage["value"] = -1
    step_button.description = STEP_LABELS[0]
    out.clear_output(wait=True)
    fig = draw_network(-1, plant_name=plant_dropdown.value)
    with out:
        display(fig)
        plt.close(fig)
        print("Weights reset to 0 and history cleared. Click Step 1 to begin a new round.")

display(widgets.HBox([w1, w2]))
display(widgets.HBox([w3, w4]))
display(b1)
display(widgets.HBox([step_button, reset_button]))
display(out)
step_button.on_click(run_step)
reset_button.on_click(reset_all)

# initial render: blank network, nothing computed yet
_fig0 = draw_network(-1, plant_name=plant_dropdown.value)
with out:
    display(_fig0)
    plt.close(_fig0)

FloatText(value=0.0, description='Bias (B1):')

Output()

## Step 5: Coach = Backpropagation (Concept)

If the diagnosis is wrong, the **Coach** revises the weights and bias and the class tries again — this is the classroom analogy for how a model recalibrates itself against lab-confirmed diagnoses.

This version gives the Coach a **calculator**: click "Suggest correction" and it computes one real gradient-descent step — the same math backpropagation performs automatically in a real network — and pre-fills the ∆ boxes below. You can apply the suggestion as-is, or edit it first to see what happens if the Coach guesses differently. Either way, scroll back up and click through **Step 1 → 2 → 3 → 4** again afterward and check the **History** table: did the Error shrink? Did the prediction flip toward the confirmed diagnosis?

*(The calculator computes, for cross-entropy loss with a sigmoid output: `∆w_i = η × (True − σ(z)) × x_i` and `∆b = η × (True − σ(z))`, where `η` is the learning rate you set. This is exactly the gradient descent update rule — see the Instructor Answer Key at the end for how it relates to real backpropagation.)*

In [3]:
#@title Coach's Calculator + Apply correction
lr_widget = widgets.FloatText(value=5.0, description='Learning rate (η):',
                               style={"description_width": "initial"})
suggest_button = widgets.Button(description='🧮 Suggest correction (gradient descent)',
                                 button_style='info', layout=widgets.Layout(width='320px'))
suggest_out = widgets.Output()

delta_w1 = widgets.FloatText(description='∆w1 (SNP):')
delta_w2 = widgets.FloatText(description='∆w2 (Path):')
delta_w3 = widgets.FloatText(description='∆w3 (Leaf):')
delta_w4 = widgets.FloatText(description='∆w4 (Root):')
delta_b1 = widgets.FloatText(description='∆Bias (B1):')

update_button = widgets.Button(description='Apply correction', button_style='success')
update_out = widgets.Output()

def suggest_correction(_):
    p = PLANTS[plant_dropdown.value]
    vals = [p["features"][f] for f in FEATURE_NAMES]
    ws = [w1.value, w2.value, w3.value, w4.value]
    bias = b1.value
    true_label = p["label"]
    z = sum(vals[i] * ws[i] for i in range(4)) + bias
    prob = sigmoid(z)
    error = true_label - prob          # True − σ(z), same Error shown above
    eta = lr_widget.value

    dw = [eta * error * vals[i] for i in range(4)]   # gradient descent step
    db = eta * error

    delta_w1.value, delta_w2.value, delta_w3.value, delta_w4.value = dw
    delta_b1.value = db

    suggest_out.clear_output()
    with suggest_out:
        print(f"Current σ(z) = {prob:.6f}   Error = {error:.6f}   η = {eta:g}")
        df = pd.DataFrame({
            "Parameter": ["∆w1 (SNP)", "∆w2 (Path)", "∆w3 (Leaf)", "∆w4 (Root)", "∆Bias"],
            "Suggested value": [round(v, 6) for v in dw] + [round(db, 6)],
        })
        display(df.style.hide(axis='index'))
        print("Suggested values have been filled in below. Edit them if you like, then click 'Apply correction'.")

def update_weights_bias(_):
    w1.value += delta_w1.value
    w2.value += delta_w2.value
    w3.value += delta_w3.value
    w4.value += delta_w4.value
    b1.value += delta_b1.value
    update_out.clear_output()
    with update_out:
        print("Weights and bias updated! Scroll up and click through Steps 1-4 again to see the effect.")

display(widgets.VBox([lr_widget, suggest_button, suggest_out]))
display(widgets.VBox([delta_w1, delta_w2, delta_w3, delta_w4, delta_b1, update_button, update_out]))
suggest_button.on_click(suggest_correction)
update_button.on_click(update_weights_bias)

## Instructor Reference — Checking the Calculator

Starting from all weights and bias at 0 on **Accession 118 — Field Trial Block 3** (confirmed Sick), `z = 0` and `σ(z) = 0.5` — exactly on the decision boundary, defaulting to an incorrect "Healthy" call. Click "Suggest correction" with the default learning rate (`η = 5`) and you should see roughly:

| | ∆w1 (SNP) | ∆w2 (Path) | ∆w3 (Leaf) | ∆w4 (Root) | ∆Bias |
|---|:---:|:---:|:---:|:---:|:---:|
| Suggested (η=5) | −0.450 | −0.500 | −1.000 | −1.500 | −2.500 |

These are proportional to each feature's value (bigger feature → bigger suggested weight change) and all share the same sign as the bias update, since `Error = True − σ(z) = −0.5` is the same for every parameter here — only the multiplying feature value differs. Apply it, and `σ(z)` should drop well below 0.5 (correct, Sick) in a single step, similar in spirit to the hand-picked reference corrections used in the manual version of this notebook — but now computed automatically, for any sample and any starting weights, by the exact update rule real training uses.

## Reflection (Discuss / Write)

1. What happened to **z** and **σ(z)** when you increased one of the weights? Which feature's weight moved the prediction the most, and does that match what you'd expect from its role in disease biology?
2. Compare Accession 42 (Greenhouse Control) and Accession 91 (Putative Tolerant Line) using the *same* final weights from your training rounds. Both are confirmed Healthy — does the model call them with similar confidence (`σ(z)`)? What does that tell you about how the model is handling *tolerance* (staying healthy despite pathogen load) versus *resistance* (low pathogen load outright)?
3. Look at the History table after applying the Coach's corrections. Did the Error shrink round over round? Did the prediction flip to match the confirmed diagnosis?
4. How is the **Coach** similar to backpropagation in a real neural network? What did the Coach do here that a real training algorithm does automatically, round after round, for thousands of samples?
5. If the Coach applied a correction with the *wrong sign* (e.g., increased a weight when it should have decreased it), what would you expect to happen to the Error on the next round?
6. Try the Suggest-correction button with a very small learning rate (η = 0.5) versus a very large one (η = 50) on the same sample. How does the suggested correction change, and what happens to σ(z) after you apply it each time? Which η actually gets Accession 118 to a correct, confident call in one step?

## Answer Key — Reflection Discussion

*(For instructor use — move, hide, or delete this cell before distributing to students if you'd rather they work it out themselves first.)*

**1. What happened to z and σ(z) when you increased a weight? Which feature moved the prediction the most?**

Increasing any single weight changes `z` by `(feature value) × (Δweight)`; since σ is monotonically increasing, a larger `z` always pushes `σ(z)` toward 1 (Healthy) and a smaller/more negative `z` pushes it toward 0 (Sick). Because feature values here range roughly 0.05–0.85, the feature with the *largest numeric value* for a given sample shows the biggest swing in `z` per unit change in its weight — e.g., for Accession 42, Root Biomass (0.80) moves `z` more than Pathogen Load (0.05) for an equal-sized weight nudge. This is a good moment to flag that a feature can dominate a model's decision simply because its raw value is large, not necessarily because it's the most biologically informative signal — exactly why real pipelines normalize/standardize inputs before training.

**2. Accession 42 (Control) vs. Accession 91 (Tolerant Line) — same confidence?**

Usually not. Accession 42 has uniformly "healthy-looking" values (high SNP diversity, very low pathogen load, high leaf color, high root biomass), so a linear model tends to call it confidently Healthy. Accession 91 has more middling values across the board — moderate pathogen load (0.30) offset by moderate (not high) vigor — so the same weights typically produce a `σ(z)` closer to the 0.5 boundary, i.e., a less confident Healthy call, even though both are truly Healthy. That's the model struggling to represent *tolerance* (staying healthy despite measurable pathogen load) versus *resistance* (low pathogen load outright) — a single-neuron linear model can't easily capture that distinction without an interaction term between pathogen load and vigor, which is exactly the kind of pattern a deeper network with hidden layers can learn that a single neuron cannot.

**3. Did the Error shrink round over round with the Calculator's corrections?**

Yes. Starting from all-zero weights on Accession 118 (confirmed Sick): Round 0 gives `σ(z) = 0.500`, Error `= 0 − 0.5 = −0.5`, and an incorrect Healthy call. Clicking **Suggest correction** at the default `η = 5` and applying it sets the weights to `[−0.45, −0.50, −1.00, −1.50]` and the bias to `−2.50`, so `z ≈ −3.98` and `σ(z) ≈ 0.018`, Error `≈ −0.018` — the call flips to Sick (correct) in a single step. Because `η = 5` is large, that one step nearly closes the gap: applying a second correction only nudges `σ(z)` down to `≈ 0.016`. If you instead use a smaller learning rate (e.g. `η = 0.5`), the Error shrinks gradually over several rounds — a clearer illustration of "round over round" convergence (see Q6). Either way the Error's magnitude shrinks monotonically, and the call flips from wrong to right after the very first correction.

**4. How is the Coach similar to backpropagation? What does a real training algorithm automate?**

The Coach and backpropagation share the same job: use the gap between prediction and ground truth to decide how each weight should move so the next prediction is closer to correct. In a real network, this isn't handed down by a person — it's three automatic pieces working together: the **loss function** plays "knows the true answer and measures how wrong we were," **backpropagation** (the chain rule applied backward through the network) plays "figures out exactly how much each individual weight contributed to that error," and the **optimizer** (Gradient Descent, Adam, etc.) plays "actually applies the correction," stepping every weight in the direction that reduces the loss, scaled by the learning rate. In this version, the Coach's Calculator *is* that math made visible: `∆w_i = η × (True − σ(z)) × x_i` is exactly one step of gradient descent on the cross-entropy loss for a single-neuron (logistic regression) model — the same formula backpropagation would compute, just derived by hand instead of by `autograd`. What the Coach still does manually here is decide *when* to click the button and *which* learning rate to use; in a real network, that loop runs automatically for every weight, every training example, over many epochs.

**5. What if the Coach's correction had the wrong sign?**

The Error would grow instead of shrink, and the prediction could become even more confidently wrong. This mirrors real training gone wrong — a bug in backpropagation's sign convention, or a learning rate so large it overshoots past the minimum — where instead of descending toward lower loss, the model climbs toward higher loss. If it compounds round after round, training diverges instead of converges, which is why sign conventions and reasonable learning rates matter in practice.


**6. Small vs. large learning rate in the Calculator:**

With η too small (0.5), the suggested correction barely moves `σ(z)` off 0.5 — technically correct direction, but it may take many rounds to actually flip the prediction, mirroring slow convergence in real training. With η too large (50), the correction overshoots dramatically (bias alone would move by −25), likely flipping `σ(z)` all the way to near 0 or oscillating past the target on later rounds if applied repeatedly — mirroring the instability an overly large learning rate causes in real training. Somewhere in between (the default η = 5 works for this example) gets Accession 118 to a correct, reasonably confident call without overshooting wildly, which is the whole reason learning-rate tuning is its own topic in practice.